# NHL Trap Games

Definition: A trap game is about a good team underperforming when they're expected to win

In [1]:
import sys
import os
import polars as pl
from tqdm import tqdm
from typing import List
import pandas as pd
import numpy as np

base_dir = '../'
sys.path.insert(1, base_dir)

import src.config as cfg
import src.utils as utils
import src.sportsipy_utils as sportsipy_utils

## Pull Schedule for Each NHL Team

Note: We are using *All Situations* as a trap game is more about the actual results on the ice than it is about a team's true underlying ability

In [2]:
# Warning: NHL Schedules not loading - https://github.com/davidjkrause/sportsipy/issues/14
# sportsipy_utils.pull_nhl_schedule('TOR', 2024)

In [3]:
# https://www.naturalstattrick.com/games.php?fromseason=20242025&thruseason=20242025&stype=2&sit=all&loc=B&team=All&rate=n
schedule_folder = f'{base_dir}data/natural_stat_trick'

df_games = []
for filename in tqdm(os.listdir(schedule_folder)):
    if 'games_' in filename:
        schedule_filepath = os.path.join(schedule_folder, filename)
        df_games_filepath = pl.scan_csv(schedule_filepath, null_values=["-"]).collect()
        if not df_games_filepath.is_empty():
            df_games += [df_games_filepath]
    
df_games = (pl.concat(df_games)
            .unique(subset=['Game', 'Team'], keep='first')
            .with_columns(
                pl.col('Game').str.split(" - ").list.get(0).str.strip_chars().str.strptime(pl.Datetime,'%Y-%m-%d').alias('Date')
            )
            .select(['Game', 'Team', 'Date', 'GF', 'GA', 'GF%', 'xGF', 'xGA', 'xGF%', 'SH%', 'SV%' ,'PDO'])
           )
utils.logger.info(f"Loaded {df_games.shape[0]/2} unique games from natural stattrick") 
df_games.head(5)

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 73.12it/s]
2025-09-13 16:45:20,957 [4006499702.py:19] [INFO] Loaded 21528.0 unique games from natural stattrick


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64
""" 2023-10-12 - Red Wings 3, Dev…","""Detroit Red Wings""",2023-10-12 00:00:00,3,4,42.86,3.42,4.87,41.24,8.57,85.19,0.938
""" 2020-03-01 - Kings 4, Golden …","""Los Angeles Kings""",2020-03-01 00:00:00,4,1,80.0,1.47,4.15,26.14,23.53,97.67,1.212
""" 2010-01-16 - Sabres 2, Island…","""Buffalo Sabres""",2010-01-16 00:00:00,2,2,50.0,2.83,4.12,40.7,6.06,94.74,1.008
""" 2024-12-21 - Penguins 0, Devi…","""New Jersey Devils""",2024-12-21 00:00:00,3,0,100.0,3.86,1.86,67.47,11.11,100.0,1.111
""" 2018-02-18 - Stars 2, Sharks …","""Dallas Stars""",2018-02-18 00:00:00,2,5,28.57,2.61,2.82,48.07,7.14,82.14,0.893


In [4]:
df_missing = df_games.filter(pl.col("GF%").is_null())
utils.logger.info(f"Found {df_missing.shape[0]} / {df_games.shape[0]} rows without any goals in all situations")

2025-09-13 16:45:20,974 [659697710.py:2] [INFO] Found 104 / 43056 rows without any goals in all situations


## Identify Trap Games

In [5]:
strength_col = "Points"
strong_threshold = 1.1
weak_threshold = 0.9

df_games_processed = (
    df_games.filter(pl.col("GF%").is_not_null())
      .unique(subset=['Game', 'Team'], keep='first')
      .sort("Game", descending=False)
      .with_columns([
        pl.when(pl.col("Date").dt.month() >= 7).then(pl.col("Date").dt.year() + 1)
            .otherwise(pl.col("Date").dt.year())
            .alias("Season"),
          pl.when(pl.col("GF%")>50).then(pl.lit(2))
              .when(pl.col("GF%")<50).then(pl.lit(0))
              .otherwise(pl.lit(1))
          .alias("Points")
      ])
    .with_columns([
        pl.col(strength_col).cum_sum().over(["Team", "Season"]).alias("Season to Date Total incl Current"),
        pl.col("GF%").cum_count().over(["Team", "Season"]).alias("# Games Season to Date incl Current")
    ])
    .with_columns(
        ((pl.col("Season to Date Total incl Current") - pl.col(strength_col))/
         (pl.col("# Games Season to Date incl Current") - 1)).alias(f"Season to Date Average {strength_col}")
    )
    .with_columns([
        ((pl.col(f"Season to Date Average {strength_col}") > strong_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_strong"),
        ((pl.col(f"Season to Date Average {strength_col}") < weak_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_weak"),

        pl.when(pl.col("Season") <= 2012).then(pl.lit("Pre-Lockout (2007–2012)"))
            .when(pl.col("Season") == 2013).then(pl.lit("2013 Lockout"))
            .when((pl.col("Season") >= 2014) & (pl.col("Season") <= 2019)).then(pl.lit("Post-Lockout (2014-2019)"))
            .when((pl.col("Season") >= 2020) & (pl.col("Season") <= 2021)).then(pl.lit("Pandemic (2020–2021)"))
            .otherwise(pl.lit("Post-Pandemic (2022–2025)"))
            .alias("Season Bucket")
    ])
)  
                     
display(df_games_processed.tail(5))
df_games_processed.mean()

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak,Season Bucket
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str
""" 2025-04-17 - Islanders 1, Blu…","""Columbus Blue Jackets""",2025-04-17 00:00:00,6,1,85.71,2.9,3.01,49.1,23.08,97.3,1.204,2025,2,74,81,0.9,false,false,"""Post-Pandemic (2022–2025)"""
""" 2025-04-17 - Lightning 0, Ran…","""Tampa Bay Lightning""",2025-04-17 00:00:00,0,4,0.0,2.17,2.22,49.48,0.0,81.82,0.818,2025,0,95,82,1.17284,true,false,"""Post-Pandemic (2022–2025)"""
""" 2025-04-17 - Lightning 0, Ran…","""New York Rangers""",2025-04-17 00:00:00,4,0,100.0,2.22,2.17,50.52,18.18,100.0,1.182,2025,2,76,81,0.925,false,false,"""Post-Pandemic (2022–2025)"""
""" 2025-04-17 - Red Wings 3, Map…","""Detroit Red Wings""",2025-04-17 00:00:00,3,4,42.86,2.73,2.42,52.95,8.82,80.0,0.888,2025,0,75,82,0.925926,false,false,"""Post-Pandemic (2022–2025)"""
""" 2025-04-17 - Red Wings 3, Map…","""Toronto Maple Leafs""",2025-04-17 00:00:00,4,3,57.14,2.42,2.73,47.05,20.0,91.18,1.112,2025,2,105,82,1.271605,true,false,"""Post-Pandemic (2022–2025)"""


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak,Season Bucket
str,str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
null,null,2016-08-21 05:52:29.413298,2.860286,2.860239,50.000698,2.768069,2.768026,50.000408,9.730112,90.27024,1.000004,2016.603511,1.000023,40.025028,40.005751,NaN,0.232981,0.225787,null


In [6]:
# Double check season derivation is reasonable given lockout in the 2012-2013 season and pandemic in 2019-2020/2020-2021 seasons
df_games_processed.group_by("Season").agg(pl.col("Game").count()).sort("Game").head(5)

Season,Game
i32,u32
2013,1434
2021,1730
2020,2160
2010,2448
2015,2448


In [7]:
df_opponent = df_games_processed.select([
    pl.col("Game"),
    pl.col("Team").alias("Opponent"),
    pl.col(f"Season to Date Average {strength_col}").alias(f"Opponent Season to Date Average {strength_col}"),
    pl.col("is_strong").alias("Opponent is_strong"),
    pl.col("is_weak").alias("Opponent is_weak")
])

df_games_with_opponent = (
    df_games_processed.join(df_opponent, on="Game", how="left")
    .filter(pl.col("Team") != pl.col("Opponent"))
    .with_columns(
        (pl.col(f"Season to Date Average {strength_col}") - pl.col(f"Opponent Season to Date Average {strength_col}")).alias(f"Delta {strength_col}")
    )

    # Tag previous opponent
    .sort(["Team", "Date"])
    .with_columns([
        pl.col(col).shift(1).over(["Team", "Season"]).alias(f"Previous {col}")
        for col in ["Opponent", "Opponent is_strong", "Opponent is_weak", "GF%"]
    ])

    # Tag Trap Games: Strong Team that beat another strong team in the previous game faces a weak opponent in the next game
    .with_columns(
        pl.when((pl.col("is_strong"))&(pl.col("Opponent is_weak"))
                &(pl.col("Previous Opponent is_strong"))&(pl.col("Previous GF%") > 50))
            .then(pl.lit(True))
        .otherwise(pl.lit(False)).alias("Trap Game")
    )
)

assert df_games_with_opponent.shape[0] == df_games_processed.shape[0]

df_games_with_opponent.filter(pl.col("Trap Game"))

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak,Season Bucket,Opponent,Opponent Season to Date Average Points,Opponent is_strong,Opponent is_weak,Delta Points,Previous Opponent,Previous Opponent is_strong,Previous Opponent is_weak,Previous GF%,Trap Game
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str,str,f64,bool,bool,f64,str,bool,bool,f64,bool
""" 2013-03-31 - Ducks 1, Blue Ja…","""Anaheim Ducks""",2013-03-31 00:00:00,1,2,33.33,1.8,1.46,55.23,3.33,88.89,0.922,2013,0,44,35,1.294118,true,false,"""2013 Lockout""","""Columbus Blue Jackets""",0.787879,false,true,0.506239,"""Chicago Blackhawks""",true,false,66.67,true
""" 2013-04-27 - Coyotes 5, Ducks…","""Anaheim Ducks""",2013-04-27 00:00:00,3,5,37.5,4.23,3.13,57.45,6.52,83.87,0.904,2013,0,57,48,1.212766,true,false,"""2013 Lockout""","""Phoenix Coyotes""",0.888889,false,true,0.323877,"""Vancouver Canucks""",true,false,75.0,true
""" 2013-12-09 - Islanders 2, Duc…","""Anaheim Ducks""",2013-12-09 00:00:00,5,2,71.43,3.51,3.57,49.62,16.13,94.59,1.107,2014,2,44,33,1.3125,true,false,"""Post-Lockout (2014-2019)""","""New York Islanders""",0.5,false,true,0.8125,"""St Louis Blues""",true,false,71.43,true
""" 2013-12-15 - Oilers 2, Ducks …","""Anaheim Ducks""",2013-12-15 00:00:00,3,2,60.0,2.59,1.6,61.77,8.82,92.0,1.008,2014,2,48,35,1.352941,true,false,"""Post-Lockout (2014-2019)""","""Edmonton Oilers""",0.617647,false,true,0.735294,"""Minnesota Wild""",true,false,66.67,true
""" 2014-01-03 - Oilers 2, Ducks …","""Anaheim Ducks""",2014-01-03 00:00:00,5,2,71.43,4.49,1.6,73.73,13.51,88.89,1.024,2014,2,62,43,1.428571,true,false,"""Post-Lockout (2014-2019)""","""Edmonton Oilers""",0.604651,false,true,0.82392,"""San Jose Sharks""",true,false,66.67,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
""" 2023-12-27 - Jets 1, Blackhaw…","""Winnipeg Jets""",2023-12-27 00:00:00,1,2,33.33,3.96,1.6,71.21,2.63,92.0,0.946,2024,0,41,33,1.28125,true,false,"""Post-Pandemic (2022–2025)""","""Chicago Blackhawks""",0.636364,false,true,0.644886,"""Boston Bruins""",true,false,83.33,true
""" 2024-04-16 - Kraken 3, Jets 4""","""Winnipeg Jets""",2024-04-16 00:00:00,4,3,57.14,2.96,1.69,63.67,16.67,86.96,1.036,2024,2,103,81,1.2625,true,false,"""Post-Pandemic (2022–2025)""","""Seattle Kraken""",0.8375,false,true,0.425,"""Colorado Avalanche""",true,false,100.0,true
""" 2025-03-16 - Jets 3, Kraken 2""","""Winnipeg Jets""",2025-03-16 00:00:00,3,2,60.0,3.23,2.79,53.66,11.54,90.48,1.02,2025,2,94,68,1.373134,true,false,"""Post-Pandemic (2022–2025)""","""Seattle Kraken""",0.895522,false,true,0.477612,"""Dallas Stars""",true,false,80.0,true


In [8]:
df_leafs_sample = df_games_with_opponent.filter((pl.col("Team")=="Toronto Maple Leafs")&(pl.col("Season")==2025))
display(df_leafs_sample.head(5))

df_leafs_sample.write_csv(f"{base_dir}data/derived/Leafs Trap Games Sample.csv")

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak,Season Bucket,Opponent,Opponent Season to Date Average Points,Opponent is_strong,Opponent is_weak,Delta Points,Previous Opponent,Previous Opponent is_strong,Previous Opponent is_weak,Previous GF%,Trap Game
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str,str,f64,bool,bool,f64,str,bool,bool,f64,bool
""" 2024-10-09 - Maple Leafs 0, C…","""Toronto Maple Leafs""",2024-10-09 00:00:00,0,1,0.0,6.09,2.97,67.2,0.0,96.3,0.963,2025,0,0,1,NaN,false,false,"""Post-Pandemic (2022–2025)""","""Montreal Canadiens""",NaN,false,false,NaN,null,null,null,null,false
""" 2024-10-10 - Maple Leafs 4, D…","""Toronto Maple Leafs""",2024-10-10 00:00:00,4,2,66.67,2.11,2.62,44.67,18.18,91.3,1.095,2025,2,2,2,0.0,false,false,"""Post-Pandemic (2022–2025)""","""New Jersey Devils""",2.0,false,false,-2.0,"""Montreal Canadiens""",false,false,0.0,false
""" 2024-10-12 - Penguins 2, Mapl…","""Toronto Maple Leafs""",2024-10-12 00:00:00,4,2,66.67,2.95,1.89,60.94,12.12,91.3,1.034,2025,2,4,3,1.0,false,false,"""Post-Pandemic (2022–2025)""","""Pittsburgh Penguins""",1.0,false,false,0.0,"""New Jersey Devils""",false,false,66.67,false
""" 2024-10-16 - Kings 2, Maple L…","""Toronto Maple Leafs""",2024-10-16 00:00:00,6,2,75.0,1.86,4.75,28.12,23.08,94.12,1.172,2025,2,6,4,1.333333,false,false,"""Post-Pandemic (2022–2025)""","""Los Angeles Kings""",0.666667,false,false,0.666667,"""Pittsburgh Penguins""",false,false,66.67,false
""" 2024-10-19 - Rangers 4, Maple…","""Toronto Maple Leafs""",2024-10-19 00:00:00,1,4,20.0,3.98,5.07,44.0,2.86,86.21,0.891,2025,0,6,5,1.5,false,false,"""Post-Pandemic (2022–2025)""","""New York Rangers""",1.5,false,false,0.0,"""Los Angeles Kings""",false,false,75.0,false


## Aggregate Tables

In [9]:
def get_agg_tables(df: pl.DataFrame, groupby_cols: List[str]) -> [pl.DataFrame, pl.DataFrame]:
    """
    
    """
    df_agg = (
        df.filter((pl.col("is_strong"))&(pl.col("Opponent is_weak"))) # compare strong vs weak games
        .group_by(groupby_cols + ['Trap Game']) # depending on whether the strong team won against another strong team the previous game
        .agg([pl.col('Game').count().alias('Sample Size').cast(pl.Int32)] 
             + [pl.col(col).mean() for col in ['GF', 'GA', 'GF%', 'xGF', 'xGA', 'xGF%', 'SH%', 'SV%', 'PDO']])
        .sort(groupby_cols + ['Trap Game'])
    )

    # Calculate % change
    value_cols = [col for col in df_agg.columns if col not in groupby_cols and col != 'Trap Game']
    if len(groupby_cols) == 0:
        df_pct_change = df_agg.filter(pl.col('Trap Game')).select(value_cols) / df_agg.filter(~pl.col('Trap Game')).select(value_cols) - 1
    else:
        df_pct_change = (df_agg
            .pivot(index=groupby_cols, on='Trap Game', values=value_cols)
            .with_columns([(pl.col(f'{col}_true') / pl.col(f'{col}_false') - 1).alias(col) 
                           for col in value_cols])
            .drop([f'{col}_true' for col in value_cols] + [f'{col}_false' for col in value_cols])
        )
    
    return df_agg, df_pct_change

In [10]:
writer = pd.ExcelWriter(f'{base_dir}data/derived/Trap Games Pivot Tables.xlsx')

for groupby_cols in [[], ['Season Bucket'], ['Season'], ['Team'], ['Season Bucket', 'Team'], ['Season', 'Team']]:
    df_agg, df_pct_change = get_agg_tables(df_games_with_opponent, groupby_cols)

    utils.logger.info(f"Pivot tables by {groupby_cols}")
    display(df_agg)
    display(df_pct_change)

    df_agg.to_pandas().to_excel(writer, sheet_name=f"Agg {'_'.join(groupby_cols)}", index=False)
    df_pct_change.to_pandas().to_excel(writer, sheet_name=f"% Change {'_'.join(groupby_cols)}", index=False)

writer.close()

2025-09-13 16:45:21,381 [902400462.py:6] [INFO] Pivot tables by []


Trap Game,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
bool,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
false,2673,3.27572,2.575383,56.401575,3.03015,2.574598,54.043471,10.447995,90.691938,1.011404
true,482,3.475104,2.522822,58.537116,3.079025,2.51471,54.922676,11.067282,90.868071,1.019342


Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.819678,0.060867,-0.020409,0.037863,0.01613,-0.023261,0.016268,0.059273,0.001942,0.007849


2025-09-13 16:45:21,402 [902400462.py:6] [INFO] Pivot tables by ['Season Bucket']


Season Bucket,Trap Game,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,bool,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2013 Lockout""",false,81,2.962963,2.54321,55.55321,2.779259,2.533827,52.320864,9.939383,90.882963,1.008235
"""2013 Lockout""",true,5,2.4,2.2,58.928,2.384,2.324,51.286,8.916,91.254,1.0016
"""Pandemic (2020–2021)""",false,314,3.273885,2.44586,58.170605,2.940924,2.463217,54.281465,10.48293,91.242771,1.017274
"""Pandemic (2020–2021)""",true,42,3.142857,2.261905,60.313095,3.04119,2.465,54.895238,9.382619,92.389524,1.017643
"""Post-Lockout (2014-2019)""",false,921,3.197611,2.541802,56.453203,2.916732,2.527101,53.632285,10.154908,91.003974,1.011587
"""Post-Lockout (2014-2019)""",true,162,3.407407,2.425926,59.674136,2.925864,2.457593,54.272222,10.828519,91.466358,1.022926
"""Post-Pandemic (2022–2025)""",false,754,3.558355,2.6313,58.038674,3.355371,2.696353,55.489602,11.237546,90.232706,1.01471
"""Post-Pandemic (2022–2025)""",true,178,3.853933,2.623596,60.234494,3.397809,2.556685,56.872865,12.27264,89.962135,1.022371
"""Pre-Lockout (2007–2012)""",false,603,3.084577,2.628524,53.468441,2.876882,2.558375,52.963967,9.958507,90.477081,1.004358


...

Season Bucket,Team,Trap Game,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,bool,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2013 Lockout""","""Anaheim Ducks""",false,9,2.111111,2.666667,45.741111,2.708889,2.66,49.795556,7.343333,91.301111,0.986556
"""2013 Lockout""","""Anaheim Ducks""",true,2,2.0,3.5,35.415,3.015,2.295,56.34,4.925,86.38,0.913
"""2013 Lockout""","""Boston Bruins""",false,10,3.1,1.7,69.889,2.904,2.679,52.148,10.421,94.417,1.0484
"""2013 Lockout""","""Carolina Hurricanes""",false,5,2.4,2.8,47.428,2.97,3.19,50.034,7.63,91.206,0.9886
"""2013 Lockout""","""Chicago Blackhawks""",false,14,3.071429,2.428571,59.434286,2.845,2.196429,56.194286,8.965714,90.451429,0.994214
…,…,…,…,…,…,…,…,…,…,…,…,…
"""Pre-Lockout (2007–2012)""","""Tampa Bay Lightning""",true,1,2.0,5.0,28.57,1.98,1.58,55.58,6.25,80.0,0.863
"""Pre-Lockout (2007–2012)""","""Vancouver Canucks""",false,50,3.06,2.34,57.2986,2.9166,2.531,53.62898,9.5616,91.7864,1.01348
"""Pre-Lockout (2007–2012)""","""Vancouver Canucks""",true,7,2.714286,2.571429,59.404286,2.551429,2.384286,52.251429,8.867143,91.444286,1.003143


Season Bucket,Team,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2013 Lockout""","""Anaheim Ducks""",-0.777778,-0.052632,0.3125,-0.225751,0.113002,-0.137218,0.131426,-0.329324,-0.0539,-0.074558
"""2013 Lockout""","""Boston Bruins""",null,null,null,null,null,null,null,null,null,null
"""2013 Lockout""","""Carolina Hurricanes""",null,null,null,null,null,null,null,null,null,null
"""2013 Lockout""","""Chicago Blackhawks""",null,null,null,null,null,null,null,null,null,null
"""2013 Lockout""","""Los Angeles Kings""",null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…
"""Pre-Lockout (2007–2012)""","""San Jose Sharks""",-0.888889,0.413223,-0.089888,0.095565,0.010888,0.068214,-0.035402,0.404548,0.016122,0.048613
"""Pre-Lockout (2007–2012)""","""St Louis Blues""",-0.909091,-0.098361,0.195652,-0.195755,-0.366014,0.729696,-0.348189,0.14714,-0.006601,0.007438
"""Pre-Lockout (2007–2012)""","""Tampa Bay Lightning""",-0.9375,-0.157895,0.860465,-0.436593,-0.289048,-0.276474,-0.007865,-0.147194,-0.104544,-0.107434


2025-09-13 16:45:21,535 [902400462.py:6] [INFO] Pivot tables by ['Season', 'Team']


Season,Team,Trap Game,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
i32,str,bool,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
2008,"""Anaheim Ducks""",false,3,3.333333,2.0,59.523333,2.403333,2.796667,45.696667,11.01,94.066667,1.051
2008,"""Boston Bruins""",false,2,3.0,3.5,41.25,2.225,2.34,47.52,9.3,87.085,0.964
2008,"""Carolina Hurricanes""",false,2,2.0,3.0,47.62,3.34,3.58,48.28,5.735,90.86,0.966
2008,"""Carolina Hurricanes""",true,1,1.0,2.0,33.33,2.55,1.55,62.29,4.17,93.75,0.979
2008,"""Dallas Stars""",false,1,5.0,2.0,71.43,4.05,2.01,66.82,12.82,90.91,1.037
…,…,…,…,…,…,…,…,…,…,…,…,…
2025,"""Vegas Golden Knights""",true,4,3.0,3.5,50.0,2.71,2.7025,49.5425,12.795,85.635,0.98425
2025,"""Washington Capitals""",false,23,3.347826,3.0,54.055652,3.303478,2.84087,53.083478,12.126957,88.042174,1.001696
2025,"""Washington Capitals""",true,6,3.166667,2.5,57.671667,2.961667,2.71,51.02,12.158333,90.016667,1.021833


Season,Team,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
i32,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2008,"""Anaheim Ducks""",null,null,null,null,null,null,null,null,null,null
2008,"""Boston Bruins""",null,null,null,null,null,null,null,null,null,null
2008,"""Carolina Hurricanes""",-0.5,-0.5,-0.333333,-0.300084,-0.236527,-0.567039,0.290182,-0.272886,0.031807,0.013458
2008,"""Dallas Stars""",null,null,null,null,null,null,null,null,null,null
2008,"""Detroit Red Wings""",-0.818182,-0.108108,-0.112903,0.277881,-0.361009,0.198718,-0.247294,0.196137,0.017472,0.033784
…,…,…,…,…,…,…,…,…,…,…,…
2025,"""Toronto Maple Leafs""",-0.555556,0.182203,-0.363208,0.318467,0.169782,-0.170663,0.201215,0.213228,0.030017,0.050832
2025,"""Vancouver Canucks""",null,null,null,null,null,null,null,null,null,null
2025,"""Vegas Golden Knights""",-0.809524,-0.148649,0.386792,-0.104936,-0.165053,0.073841,-0.112747,0.140471,-0.041021,-0.020925


## Permutation Tests

- Test Statistic: Difference in $GF\%$ between trap games and similar non-trap games  
- $H_0$: No difference between trap games and similar non-trap games. Any difference due to random chance alone
- $H_1$: Teams perform worse in trap games, i.e., Test Statistic < 0

In [11]:
def get_pvalues(df: pl.DataFrame, groupby_cols: List[str], num_simulations: int=1000) -> pl.DataFrame:
    """
    Run a one-sided permutation test on alternative hypothesis that teams perform worse in trap games on a GF% basis
    :param df: pl.DataFrame
    :param groupby_cols: List[str]
    :param num_simulations: int
    :return: pl.DataFrame
    """
    # Observed Test Stat
    _, df_pct_change_obs = get_agg_tables(df, groupby_cols)
    df_obs = df_pct_change_obs.select(groupby_cols + ['GF%']).rename({'GF%': 'Observed GF%'})
    
    # Derive Simulated Test Stats
    sim_results = []
    for i in tqdm(range(num_simulations)):
        df_permuted_all = []
        for category, df_group in df.group_by(groupby_cols, maintain_order=True):
            # Permute 'GF%' labels among all similar games
            df_permuted = df_group.filter((pl.col("is_strong"))&(pl.col("Opponent is_weak"))) # compare strong vs weak games
            permuted_values = np.random.permutation(df_permuted['GF%'].to_numpy())
            df_permuted = df_permuted.with_columns(pl.Series('GF%', permuted_values))
            df_permuted_all.append(df_permuted)
        
        df_permuted_all = pl.concat(df_permuted_all)
        _, df_pct_change_perm = get_agg_tables(df_permuted_all, groupby_cols)
        
        sim_results.append(
            df_pct_change_perm.select(groupby_cols + ['GF%'])
            .rename({'GF%': 'Simulated GF%'})
            .with_columns(pl.lit(i).alias("Simulation"))
        )
    
    # one row per groupby_col x simulation # x simulated test stat
    df_simulated = pl.concat(sim_results)
    df_test_stats = df_simulated.join(df_obs, on=groupby_cols, how="left")
    
    # One-sided p-value where observed test stats are further in the direction of the alternative
    df_pvals = (
        df_test_stats
        .with_columns(
            (pl.col("Simulated GF%") <= pl.col("Observed GF%")).cast(pl.Int32).alias("Extreme")
        )
        .group_by(groupby_cols)
        .agg([
            pl.mean("Extreme").alias("P-Value"),
            pl.first("Observed GF%"),   # keep observed stat
        ])
    ).filter(pl.col('P-Value').is_not_null()).sort('P-Value')

    return df_pvals

In [ ]:
writer = pd.ExcelWriter(f'{base_dir}data/derived/Trap Games P-Values.xlsx')

for groupby_cols in [['Season'], ['Team'], ['Season Bucket', 'Team']]:
    utils.logger.info(groupby_cols)
    df_pvals = get_pvalues(df_games_with_opponent, groupby_cols, 10000)
    display(df_pvals)

    df_pvals.to_pandas().to_excel(writer, sheet_name=f"P-Value {'_'.join(groupby_cols)}", index=False)

writer.close()

2025-09-13 16:45:21,849 [643730260.py:4] [INFO] ['Season']
  1%|▌                                                                                        | 60/10000 [00:01<03:00, 55.22it/s]